# Kaggle V3: Full Training Pipeline (Fixed Chat Template)
> **Changes in V3:**
> - Bypasses Gemma's chat template limitations by explicitly serializing `<tool_call>` actions into text.
> - Removes `max_steps=60` limit (now trains for 1 full epoch on Kaggle).
> - Fixes `inspect-evals` Docker crash with `--sandbox local`.

## 1. Install Dependencies

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl peft accelerate bitsandbytes transformers datasets huggingface_hub trackio inspect-ai inspect-evals
print("✅ Dependencies installed.")

## 2. Configuration & Authentication

In [ ]:
import os
from pathlib import Path
from huggingface_hub import HfApi, login

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = input('Paste your HF write token: ').strip()

os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)

HF_USERNAME = HfApi().whoami()['name']
DATASET_ID = "badlogicgames/pi-mono"
MODEL_ID = "unsloth/gemma-2-2b-it-bnb-4bit"
FINAL_REPO_ID = f"{HF_USERNAME}/fine-tuning-agent-on-traces-v3"
TRACKIO_PROJECT = "agent-fine-tuning-on-trace-v3"
MAX_SEQ_LENGTH = 2048

print(f"✅ Configured. Final model will be: {FINAL_REPO_ID}")

## 3. Dataset Preprocessing (Fixed Tool Calling)

In [ ]:
import hashlib, json, random
from datasets import Dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

print("Downloading traces...")
raw_dir = Path("./pi_mono_raw")
snapshot_download(repo_id=DATASET_ID, repo_type="dataset", allow_patterns=["*.jsonl"], local_dir=str(raw_dir))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def extract_text(parts, max_chars=12000):
    if isinstance(parts, str): return parts[:max_chars]
    if not isinstance(parts, list): return ""
    out = []
    for p in parts:
        if isinstance(p, dict) and p.get("type") == "text":
            out.append(str(p.get("text") or "").strip())
    return "\n".join(out)[:max_chars]

def raw_to_chat(event):
    if event.get("type") != "message": return None
    raw = event.get("message") or {}
    role = raw.get("role")
    
    if role == "user":
        return {"role": "user", "content": extract_text(raw.get("content"))}
        
    if role == "assistant":
        parts = raw.get("content") or []
        text = extract_text(parts)
        tc_strs = []
        if isinstance(parts, list):
            for p in parts:
                if isinstance(p, dict) and p.get("type") == "toolCall":
                    tc = {"name": p.get("name"), "arguments": p.get("arguments") or {}}
                    tc_strs.append(json.dumps(tc))
        if tc_strs:
            text += "\n<tool_call>\n" + "\n".join(tc_strs) + "\n</tool_call>"
        return {"role": "assistant", "content": text.strip()} if text.strip() else None
        
    if role == "toolResult":
        name = raw.get("toolName", "unknown")
        content = extract_text(raw.get("content")) or "[empty]"
        return {"role": "user", "content": f"<tool_result name=\"{name}\">\n{content}\n</tool_result>"}
    return None

examples = []
for path in raw_dir.glob("*.jsonl"):
    events = [json.loads(line) for line in path.read_text(errors="replace").splitlines() if line.strip()]
    conv = []
    for e in events:
        msg = raw_to_chat(e)
        if not msg: continue
        if conv and conv[-1]["role"] == msg["role"]:
            conv[-1]["content"] += "\n\n" + msg["content"]
        else:
            conv.append(msg)
            
    for i in range(1, len(conv)):
        if conv[i]["role"] == "assistant":
            try:
                prompt = tokenizer.apply_chat_template(conv[:i], tokenize=False, add_generation_prompt=True)
                full = tokenizer.apply_chat_template(conv[:i+1], tokenize=False, add_generation_prompt=False)
                if full.startswith(prompt):
                    comp = full[len(prompt):]
                    if len(comp.strip()) > 5 and len(tokenizer(prompt+comp)["input_ids"]) <= MAX_SEQ_LENGTH:
                        examples.append({"prompt": prompt, "completion": comp})
            except Exception: pass

random.Random(42).shuffle(examples)
eval_size = max(1, len(examples) // 20)
train_ds = Dataset.from_list(examples[eval_size:])
eval_ds = Dataset.from_list(examples[:eval_size])
print(f"✅ Built {len(examples)} examples (Train: {len(train_ds)}, Eval: {len(eval_ds)})")

## 4. Train (1 Full Epoch)

In [ ]:
import gc, torch, trackio
from peft import LoraConfig
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

job_id = "job_01_v3_full_epoch"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
model.config.use_cache = False

peft_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], task_type="CAUSAL_LM")

trackio.init(project=TRACKIO_PROJECT, name=job_id, config={"epochs": 1})

sft_config = SFTConfig(
    output_dir=f"./results/{job_id}",
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,
    dataset_text_field=None,
    learning_rate=1e-4,
    num_train_epochs=1,     # <--- Full epoch instead of max_steps=60!
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    report_to="none",
)

trainer = SFTTrainer(model=model, args=sft_config, train_dataset=train_ds, eval_dataset=eval_ds, peft_config=peft_config, processing_class=tokenizer)
trainer.train()

trackio.log({"train_loss": trainer.state.log_history[-1].get("train_loss", 0)})
trackio.finish()

trainer.save_model(sft_config.output_dir)
del model, trainer
gc.collect(); torch.cuda.empty_cache()

## 5. Merge & Push

In [ ]:
from peft import PeftModel
print("Merging weights...")
base_model = AutoModelForCausalLM.from_pretrained("google/gemma-2-2b-it", dtype=torch.float16, device_map="auto")
merged = PeftModel.from_pretrained(base_model, f"./results/{job_id}").merge_and_unload()
merged.save_pretrained("./final_model", safe_serialization=True)
tokenizer.save_pretrained("./final_model")

print("Pushing...")
HfApi().create_repo(FINAL_REPO_ID, token=HF_TOKEN, exist_ok=True)
HfApi().upload_folder(folder_path="./final_model", repo_id=FINAL_REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(FINAL_REPO_ID, token=HF_TOKEN)
del base_model, merged
gc.collect(); torch.cuda.empty_cache()

## 6. Eval & Before/After

In [ ]:
import textwrap
print("Running Evals...")
!inspect eval inspect_evals/humaneval --model hf/{FINAL_REPO_ID} --limit 50 --log-dir ./logs --max-tokens 512 --sandbox local

print("\n--- BEFORE VS AFTER ---")
def infer(m_id): 
    m = AutoModelForCausalLM.from_pretrained(m_id, quantization_config=bnb_config, device_map="auto")
    t = AutoTokenizer.from_pretrained(m_id)
    p = t.apply_chat_template([{"role":"user", "content":"Read README.md"}], tokenize=False, add_generation_prompt=True)
    out = m.generate(**t(p, return_tensors="pt").to(m.device), max_new_tokens=150)
    return t.decode(out[0][len(t(p)["input_ids"]):], skip_special_tokens=True)

print("🔵 BASE:\n", textwrap.fill(infer("google/gemma-2-2b-it")))
gc.collect(); torch.cuda.empty_cache()
print("\n🟢 V3 AGENT:\n", textwrap.fill(infer(FINAL_REPO_ID)))
gc.collect(); torch.cuda.empty_cache()